# ChargeGrid Intelligence — Sprint 4 · avaliação no Colab

Roda o projeto inteiro no Colab, sem instalar nada no computador:

1. clona o repositório e instala as dependências;
2. indexa a base de conhecimento no ChromaDB (`nomic-embed-text`);
3. roda as avaliações — RAGAS nas 3 iterações, a versão das Sprints 1/2, a comparação de
   modelos e os testes de segurança;
4. preenche as tabelas dos relatórios e gera `docs/relatorio_evolucao.pdf`;
5. (opcional) sobe a interface web com link público.

**Antes de rodar:** no menu à esquerda, clique no ícone de chave (**Secrets**) e cadastre:

| Nome | Onde conseguir |
|---|---|
| `GROQ_API_KEY` | <https://console.groq.com/keys> |
| `OLLAMA_API_KEY` | <https://ollama.com> → Settings → Keys |
| `GITHUB_TOKEN` *(opcional)* | só para enviar os resultados de volta ao GitHub (seção 6) |

Ligue o acesso do notebook a cada segredo. Depois use **Ambiente de execução → Executar tudo**.
A rodada completa leva de 30 a 60 minutos por causa do limite da conta gratuita do Groq.

## 1. Projeto e dependências

In [ ]:
import os

REPO = "https://github.com/pedrosampaiom2007-gif/sprint-4-Jorge.git"
PASTA = "/content/sprint-4-Jorge"

if not os.path.exists(PASTA):
    !git clone -q {REPO} {PASTA}
else:
    !git -C {PASTA} pull -q
%cd {PASTA}
!pip install -q -r requirements.txt

## 2. Chaves

In [ ]:
from google.colab import userdata

for nome in ("GROQ_API_KEY", "OLLAMA_API_KEY"):
    os.environ[nome] = userdata.get(nome)
print("chaves carregadas:", [n for n in ("GROQ_API_KEY", "OLLAMA_API_KEY") if os.environ.get(n)])

## 3. Embeddings (nomic-embed-text)

A célula testa o `nomic-embed-text` na Ollama Cloud. Se a sua conta não servir esse modelo,
ela instala o Ollama dentro do Colab e usa o modelo local — o resto do notebook não muda.

In [ ]:
import subprocess, time

def testar_embedding():
    from src.rag.embeddings import criar_embeddings
    return len(criar_embeddings().embed_query("teste de recarga"))

try:
    print("Ollama Cloud ok, dimensão:", testar_embedding())
except Exception as erro:
    print("Ollama Cloud não serviu o nomic-embed-text:", str(erro)[:200])
    print("Instalando o Ollama no Colab...")
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    !ollama pull nomic-embed-text
    os.environ["OLLAMA_EMBED_BASE_URL"] = "http://localhost:11434"
    print("Ollama local ok, dimensão:", testar_embedding())

## 4. Indexação e testes offline

In [ ]:
!python -m src.rag.vector_store
!python -m unittest discover -s tests

In [ ]:
# relevância do melhor trecho nas perguntas com e sem resposta na base
!python -m evals.calibrar_limiar

## 5. Avaliações

Cada célula grava um JSON em `evals/resultados/`. Se uma cair no meio (limite da Groq),
rode a mesma célula de novo.

In [ ]:
# coluna "antes": o chatbot das Sprints 1/2 no mesmo eval set
!python -m evals.run_legado_rag

In [ ]:
# iterações 1, 2 e 3 do RAG com RAGAS (faithfulness e answer_relevancy)
!python -m evals.run_iteracoes

In [ ]:
# gpt-oss-20b x gpt-oss-120b (Groq) x gemma4:cloud (Ollama)
!python -m evals.run_modelos

In [ ]:
# prompt injection escondido em documento, com e sem a blindagem
!python -m evals.seguranca_documentos

In [ ]:
# bateria de segurança da Sprint 3 (24 casos, 12 ataques)
!python -m evals.run_evals

## 6. Relatórios e PDF

In [ ]:
!python -m evals.consolidar > /dev/null
from IPython.display import Markdown, display
display(Markdown(open("evals/historico_scores.md", encoding="utf-8").read()))

In [ ]:
# baixa resultados, relatórios e o PDF
!zip -qr resultados_sprint4.zip evals/resultados evals/historico_scores.md docs prompts/rag/CHANGELOG.md
from google.colab import files
files.download("resultados_sprint4.zip")
files.download("docs/relatorio_evolucao.pdf")

### Enviar os resultados para o GitHub *(opcional)*

Precisa do segredo `GITHUB_TOKEN` (GitHub → Settings → Developer settings → Personal access
tokens, com permissão de escrita no repositório). Preencha nome e e-mail do autor dos commits.

In [ ]:
NOME_AUTOR = "Pedro Sampaio"
EMAIL_AUTOR = "pedrosampaiom2007@gmail.com"

token = userdata.get("GITHUB_TOKEN")
!git config user.name "{NOME_AUTOR}"
!git config user.email "{EMAIL_AUTOR}"
!git add evals/resultados evals/historico_scores.md docs prompts/rag/CHANGELOG.md
!git commit -q -m "Resultados das avaliações da Sprint 4 (RAGAS, modelos e segurança)"
!git push -q https://{token}@github.com/pedrosampaiom2007-gif/sprint-4-Jorge.git HEAD:main

## 7. Interface web *(opcional)*

In [ ]:
# gera um link público do Gradio (válido enquanto o notebook estiver rodando)
from app.web import construir_interface
construir_interface().launch(share=True)